# GeoShapley

## I. Stacking model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, make_scorer, cohen_kappa_score, RocCurveDisplay

from sklearn.preprocessing import MinMaxScaler, RobustScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, ElasticNet
from sklearn.ensemble import StackingClassifier, GradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor

import lightgbm as lgb
import xgboost as xgb

from sklearn.inspection import PartialDependenceDisplay, partial_dependence, permutation_importance

from sklearn.datasets import load_iris, make_moons


In [ ]:
in_zero_df_month = pd.read_csv(r'D:/Data/in_zero_df_month.csv')
non_zero_df_month = pd.read_csv(r'D:/Data/non_zero_df_month.csv')

In [ ]:
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_wmape(y_true, y_pred):
    sum_abs_errors = np.sum(np.abs(y_true - y_pred))
    sum_actuals = np.sum(y_true)
    if sum_actuals == 0:
        return np.nan
    else:
        wmape = sum_abs_errors / sum_actuals
        return wmape

In [ ]:
data = non_zero_df_month

In [ ]:
data = data.drop('Unnamed: 0', axis=1)

In [ ]:
data = data.drop('Polygon_ID', axis=1)

In [ ]:
id = data['IDCode']

In [ ]:
data['grid_id'] = data['IDCode'].factorize()[0] + 1
re_data = data.dropna()

In [ ]:
re_data['month_sin'] = np.sin(2 * np.pi * re_data['Month'] / 12)
re_data['month_cos'] = np.cos(2 * np.pi * re_data['Month'] / 12)

In [ ]:
def add_lag_features_by_idcode(df, lag=1, columns=[], id_column='grid_id'):
    for col in columns:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)
    return df

pollutants = ['CO_MEAN', 'NO2_MEAN', 'PM2_5_MEAN', 'PM10_MEAN', 'SO2_MEAN', 'LSTA_MEAN', 'LSTT_MEAN', 'NTL_MEAN', 'O3_MEAN'] 
for lag in range(1, 4):  # Adding lag1, lag2, lag3
    re_data = add_lag_features_by_idcode(re_data, lag=lag, columns=pollutants)

In [ ]:
re_data = re_data.dropna()

In [ ]:
X = X.drop(['IDCode'], axis=1)

In [ ]:
X = re_data.drop(['GridProd_Steel_tot', 'GridProd_Iron_tot', 'grid_id'], axis=1)
y_needed_steel = np.log(re_data['GridProd_Steel_tot'])

## I.A Steel Output

## (1) Stacking and NN

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

X_train, X_test, y_train, y_test = train_test_split(X, y_needed_steel, test_size=0.2, random_state=1)

pollutants = ['CO_MEAN', 'NO2_MEAN', 'PM2_5_MEAN', 'PM10_MEAN', 
              'SO2_MEAN', 'LSTA_MEAN', 'LSTT_MEAN', 'NTL_MEAN', 'O3_MEAN']
other_features = [col for col in X.columns if col not in pollutants + ['Centroid_Lat', 'Centroid_Long']]


preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants),
        ('spatial', RobustScaler(), ['Centroid_Long', 'Centroid_Lat']),
        ('others', StandardScaler(), other_features)
    ])

base_models = [
    ('lasso', Lasso(alpha=0.01, random_state=1)),
    ('lightgbm', lgb.LGBMRegressor(objective='regression', num_leaves=5, learning_rate=0.05, n_estimators=720)),
    ('random_forest', RandomForestRegressor(max_depth=4, max_features=9, n_estimators=300, random_state=5)),
    ('xgboost', xgb.XGBRegressor(colsample_bytree=0.6, learning_rate=0.1, max_depth=6, n_estimators=300, random_state=5))
]

meta_model = LinearRegression()
stacking_model = StackingRegressor(estimators=base_models, final_estimator=meta_model, cv=5)

models = {
    "Stacking Model": make_pipeline(preprocessor, stacking_model),
    "Neural Network": make_pipeline(preprocessor, MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=500, random_state=1))
}

results_steel = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results_steel.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": calculate_rmse(y_test, y_pred),
        "MAPE": calculate_mape(y_test, y_pred),
        "WMAPE": calculate_wmape(y_test, y_pred),
        "R2 Score": r2_score(y_test, y_pred)
    })


results_df_steel = pd.DataFrame(results_steel)


In [ ]:
results_df_steel

## (b) KFold approach: XGBoost

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd


X_train, X_test, y_train, y_test = train_test_split(X, y_needed_steel, test_size=0.2, random_state=1)

IDCode_train = X_train['IDCode']
IDCode_test = X_test['IDCode']

X_train = X_train.drop(['IDCode'], axis=1)
X_test = X_test.drop(['IDCode'], axis=1)

pollutants = ['CO_MEAN', 'NO2_MEAN', 'PM2_5_MEAN', 'PM10_MEAN', 
              'SO2_MEAN', 'LSTA_MEAN', 'LSTT_MEAN', 'NTL_MEAN', 'O3_MEAN']

other_features = [col for col in X_train.columns if col not in pollutants + ['Centroid_Lat', 'Centroid_Long']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants),  # Standardize pollutants
        ('spatial', RobustScaler(), ['Centroid_Long', 'Centroid_Lat']),  # Robust scaling for spatial features
        ('others', StandardScaler(), other_features)  # Standardize other features
    ])

xgb_model = make_pipeline(preprocessor, xgb.XGBRegressor(
    colsample_bytree=0.6,
    learning_rate=0.1,
    max_depth=6,
    n_estimators=300,
    subsample=0.8,
    alpha=0.2,  
    reg_lambda=0.5,  
    random_state=5
))

kf = KFold(n_splits=5, shuffle=True, random_state=1)

mse_scores = cross_val_score(xgb_model, X, y_needed_steel, cv=kf, scoring=make_scorer(mean_squared_error))
rmse_scores = cross_val_score(xgb_model, X, y_needed_steel, cv=kf, scoring=make_scorer(calculate_rmse))
mape_scores = cross_val_score(xgb_model, X, y_needed_steel, cv=kf, scoring=make_scorer(calculate_mape))
wmape_scores = cross_val_score(xgb_model, X, y_needed_steel, cv=kf, scoring=make_scorer(calculate_wmape))
r2_scores = cross_val_score(xgb_model, X, y_needed_steel, cv=kf, scoring='r2')

results = {
    "Metric": ["MSE", "RMSE", "MAPE", "WMAPE", "R²"],
    "Mean": [
        np.mean(mse_scores),
        np.mean(rmse_scores),
        np.mean(mape_scores),
        np.mean(wmape_scores),
        np.mean(r2_scores)
    ],
    "Std Dev": [
        np.std(mse_scores),
        np.std(rmse_scores),
        np.std(mape_scores),
        np.std(wmape_scores),
        np.std(r2_scores)
    ]
}

In [ ]:
results_df_steel = pd.DataFrame(results)

print(results_df_steel)

## SHAP

In [ ]:
import shap

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
xgb_model.score(X_test, y_test)
y_pred = xgb_model.predict(X_test)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(12, 6))


plt.subplot(1, 2, 1)
sns.kdeplot(y_test, color='blue', fill=True, label='Actual')
sns.kdeplot(y_pred, color='green', fill=True, label='Predicted')
plt.title('Density of Actual vs Predicted Values')
plt.xlabel('Values (10,000 tones)')
plt.legend()


original_ticks = [0.01, 0.1, 1, 10, 100]
log_ticks = [np.log(tick) for tick in original_ticks] 
plt.xticks(log_ticks, original_ticks)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred, alpha=0.5, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs Predicted Values')
plt.xlabel('Actual Values (10,000 tones)')
plt.ylabel('Predicted Values (10,000 tones)')


plt.xticks(log_ticks, original_ticks) 
plt.yticks(log_ticks, original_ticks) 

plt.tight_layout()
plt.savefig(r'D:\Figures\predicted_grid_steel_distribution_adj_realnumber.pdf', format='pdf', dpi=800)
plt.show()

In [ ]:
xgb_model_only = xgb_model.named_steps['xgbregressor']

explainer_shap = shap.TreeExplainer(xgb_model_only)

shap_values = explainer_shap.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

## Compute confidence intervals for SHAPs

In [ ]:

from tqdm import tqdm
import numpy as np
import shap
import pandas as pd

def bootstrap_shap(xgb_model, X_train, y_train, X_test, n_bootstraps=5000):
   
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    y_pred = xgb_model.predict(X_train)
    y_pred = pd.Series(y_pred, index=y_train.index)  
    err = y_train - y_pred

    shap_bootstrap_list = []

    xgboost_model = xgb_model.named_steps['xgbregressor']
    explainer_shap = shap.TreeExplainer(xgboost_model)

    for i in tqdm(range(n_bootstraps), desc="Bootstrapping SHAP Values"):
        random_sample_index = np.random.choice(y_train.index, size=len(y_train), replace=True)

        y_sample = y_pred + err.loc[random_sample_index].values

        xgboost_model.fit(X_train, y_sample)

        X_test_transformed = xgb_model.named_steps['columntransformer'].transform(X_test)

        shap_values = explainer_shap.shap_values(X_test_transformed)
        shap_bootstrap_list.append(shap_values)

    return np.array(shap_bootstrap_list)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

shap_bootstrap_values = bootstrap_shap(xgb_model, X_train, y_train, X_test)

In [ ]:
shap_bootstrap_list = shap_bootstrap_values

In [ ]:
shap_bootstrap_values.shape

## Compute the confidence intervals for the global shap values

In [ ]:
mean_shap_global = np.abs(shap_bootstrap_list).mean(axis=1).mean(axis=0)

lower_percentile = np.percentile(np.abs(shap_bootstrap_list).mean(axis=1), axis=0, q=2.5)
upper_percentile = np.percentile(np.abs(shap_bootstrap_list).mean(axis=1), axis=0, q=97.5)

l_shap_global = np.maximum(0, mean_shap_global - lower_percentile)
u_shap_global = np.maximum(0, upper_percentile - mean_shap_global)

df_mean_shap = pd.DataFrame(
    np.vstack([
        np.array(X_train.columns), 
        mean_shap_global,        
        l_shap_global,              
        u_shap_global,            
        lower_percentile,           
        upper_percentile            
    ]).T,
    columns=['Feature', 'SHAP', 'SHAP_err_l', 'SHAP_err_u', 'SHAP_l', 'SHAP_u']
)

df_mean_shap[['SHAP', 'SHAP_err_l', 'SHAP_err_u', 'SHAP_l', 'SHAP_u']] = \
    df_mean_shap[['SHAP', 'SHAP_err_l', 'SHAP_err_u', 'SHAP_l', 'SHAP_u']].astype(float)

print(df_mean_shap)


## Global feature importance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_mean_shap_sorted = df_mean_shap.sort_values(by='SHAP', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10), dpi=160) 

ax.barh(
    df_mean_shap_sorted.Feature,
    df_mean_shap_sorted.SHAP,
    xerr=df_mean_shap_sorted[['SHAP_err_l', 'SHAP_err_u']].values.T,
    color=shap.plots.colors.blue_rgb  
)

ax.set_xlabel('SHAP (average impact on the outcome)', fontsize=14)  
ax.set_ylabel('Features', fontsize=14)  

ax.set_yticks(np.arange(len(df_mean_shap_sorted.Feature))) 
ax.set_yticklabels(
    df_mean_shap_sorted.Feature, 
    fontsize=12,  
    va='center' 
)

plt.subplots_adjust(left=0.3)  
plt.tight_layout()  

plt.savefig(r'D:\Figures\SHAP.pdf', format='pdf', dpi=800)
plt.show()


## Visualize the confidence intervals for the local shap values

In [ ]:
print(shap_bootstrap_list.shape)

In [ ]:

l_95 = np.percentile(shap_bootstrap_list,axis=0,q=2.5)
u_95 = np.percentile(shap_bootstrap_list,axis=0,q=97.5)

In [ ]:
import seaborn as sns

def plot(ax,term=0):
    
    order = np.argsort(X_train.values[:,term])

    ax.fill_between(X_train.values[:,term][order], l_95[:,term][order], 
                     u_95[:,term][order],color='lightblue',alpha=0.6)
    
    ax.scatter(X_train.values[:,term][order], shap_bootstrap_list.mean(axis=0)[:,term][order],
                s=10,color='black')
    
    ax.axhline(0,color='r', linestyle='--',)
    ax.set_xlabel(X_names[term],fontsize=13)
    ax.set_ylabel("SHAP value",fontsize=13)
    plt.tight_layout()

## Partial dependence plots

In [ ]:
print("X_train shape:", X_train.shape)  
print("l_95 shape:", l_95.shape)       
print("u_95 shape:", u_95.shape)       
print("shap_bootstrap_list shape:", shap_bootstrap_list.shape)
print("X_test shape:", X_train.shape) 

In [ ]:
X_subset = X_train.iloc[:5798]  

if isinstance(X_train, pd.DataFrame):
    X_names = X_train.columns.tolist() 
else:
    X_names = [f"Feature {i}" for i in range(X_train.shape[1])]
    
def plot(ax, term=0):
    order = np.argsort(X_subset.values[:, term])

    ax.fill_between(
        X_subset.values[:, term][order], 
        l_95[:, term][order], 
        u_95[:, term][order], 
        color='lightblue', 
        alpha=0.6
    )

    ax.scatter(
        X_subset.values[:, term][order], 
        shap_bootstrap_list.mean(axis=0)[:, term][order], 
        s=10, 
        color='black'
    )

    ax.axhline(0, color='r', linestyle='--')
    ax.set_xlabel(X_names[term], fontsize=13)
    ax.set_ylabel("SHAP value", fontsize=13)
    ax.set_title(f"Feature: {X_names[term]}", fontsize=10)

fig, ax = plt.subplots(3, 3, figsize=(12, 8), dpi=300)
ax = ax.ravel()
selected_features = [2, 7, 9, 8, 3, 5, 10, 4, 6]

for index, j in enumerate(selected_features):
    plot(ax=ax[index], term=j)

plt.tight_layout()
plt.savefig(r'D:\Figures\PDP.pdf', format='pdf', dpi=800)
plt.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

X_train, X_test, y_train, y_test = train_test_split(X, y_needed_steel, test_size=0.2, random_state=1)

pollutants = ['CO_MEAN', 'NO2_MEAN', 'PM2_5_MEAN', 'PM10_MEAN', 
              'SO2_MEAN', 'LSTA_MEAN', 'LSTT_MEAN', 'NTL_MEAN', 'O3_MEAN']

other_features = [col for col in X.columns if col not in pollutants + ['Centroid_Lat', 'Centroid_Long']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants),
        ('spatial', RobustScaler(), ['Centroid_Long', 'Centroid_Lat']),
        ('others', StandardScaler(), other_features)
    ])

base_models = [
    ('lasso', Lasso(alpha=0.01, random_state=1)),
    ('lightgbm', lgb.LGBMRegressor(objective='regression', num_leaves=5, learning_rate=0.05, n_estimators=720)),
    ('random_forest', RandomForestRegressor(max_depth=4, max_features=9, n_estimators=300, random_state=5)),
    ('xgboost', xgb.XGBRegressor(colsample_bytree=0.6, learning_rate=0.1, max_depth=6, n_estimators=300, random_state=5))
]

meta_model = LinearRegression()
stacking_model = StackingRegressor(estimators=base_models, final_estimator=meta_model, cv=5)

models = {
    "Stacking Model": make_pipeline(preprocessor, stacking_model),
    "Neural Network": make_pipeline(preprocessor, MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=500, random_state=1))
}

results_steel = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results_steel.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": calculate_rmse(y_test, y_pred),
        "MAPE": calculate_mape(y_test, y_pred),
        "WMAPE": calculate_wmape(y_test, y_pred),
        "R2 Score": r2_score(y_test, y_pred)
    })

results_df_steel = pd.DataFrame(results_steel)

results_df_steel

In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
import xgboost as xgb


def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


xgboost_model = make_pipeline(
    preprocessor, 
    xgb.XGBRegressor(
        colsample_bytree=0.6,  
        learning_rate=0.1,     
        max_depth=6,           
        n_estimators=300,     
        subsample=0.8,         
        alpha=0.2,         
        reg_lambda=0.5,     
        random_state=5       
    )
)

kf = KFold(n_splits=5, shuffle=True, random_state=1)
xgboost_rmse_scores = []

final_model = None
X_test_fold_final = None
y_test_fold_final = None

for train_idx, test_idx in kf.split(X):
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y_needed_steel.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]
    y_test_fold = y_needed_steel.iloc[test_idx]
    
    xgboost_model.fit(X_train_fold, y_train_fold)
    y_pred_fold = xgboost_model.predict(X_test_fold)
    rmse_fold = calculate_rmse(y_test_fold, y_pred_fold)
    xgboost_rmse_scores.append(rmse_fold)
    
    final_model = xgboost_model
    X_test_fold_final = X_test_fold
    y_test_fold_final = y_test_fold

print(f"Average RMSE across folds: {np.mean(xgboost_rmse_scores):.4f}")


xgboost_regressor = final_model.named_steps['xgbregressor']

In [ ]:
import shap
import matplotlib.pyplot as plt


explainer = shap.Explainer(xgboost_regressor, preprocessor.transform(X_train_fold))


shap_values = explainer(preprocessor.transform(X_test_fold_final))


plt.figure()  
shap.summary_plot(
    shap_values, 
    preprocessor.transform(X_test_fold_final), 
    feature_names=X.columns,
    show=False  
)

plt.tight_layout()  
output_path = r'D:\Figures\SHAP_grid.pdf'
plt.savefig(output_path, format='pdf', dpi=800)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from tqdm import tqdm

def bootstrap_shap(pipeline_model, X, y_train, X_test, n_bootstraps=50, save_path=None):
   
    X = X.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    preprocessor = pipeline_model.named_steps['columntransformer']
    xgb_regressor = pipeline_model.named_steps['xgbregressor']

    explainer_shap = shap.TreeExplainer(xgb_regressor)

    X_transformed = preprocessor.transform(X)
    X_test_transformed = preprocessor.transform(X_test)

    shap_bootstrap_list = []

    for i in tqdm(range(n_bootstraps), desc="Bootstrapping SHAP Values"):
        sample_indices = np.random.choice(range(len(X)), size=len(X), replace=True)
        X_sampled = X_transformed[sample_indices]
        y_sampled = y_train.iloc[sample_indices]

        xgb_regressor_clone = xgb_regressor.__class__(**xgb_regressor.get_params())
        xgb_regressor_clone.fit(X_sampled, y_sampled)

        explainer_shap_clone = shap.TreeExplainer(xgb_regressor_clone)
        shap_values = explainer_shap_clone.shap_values(X_test_transformed)
        shap_bootstrap_list.append(shap_values)

        if save_path and (i + 1) % 10 == 0:
            np.save(save_path, np.array(shap_bootstrap_list))

    shap_bootstrap_array = np.array(shap_bootstrap_list)

    if save_path:
        np.save(save_path, shap_bootstrap_array)

    return shap_bootstrap_array


save_path = "shap_bootstrap_intermediate.npy"
shap_bootstrap_values = bootstrap_shap(final_model, X, y_needed_steel, X_test_fold_final, n_bootstraps=50, save_path=save_path)

mean_shap = np.abs(shap_bootstrap_values).mean(axis=1).mean(axis=0)
lower_bound = mean_shap - np.percentile(np.abs(shap_bootstrap_values).mean(axis=1), q=2.5, axis=0)
upper_bound = np.percentile(np.abs(shap_bootstrap_values).mean(axis=1), q=97.5, axis=0) - mean_shap

df_mean_shap = pd.DataFrame(
    {
        "Feature": np.array(X.columns),
        "Mean SHAP": mean_shap,
        "Lower Error": np.maximum(0, lower_bound), 
        "Upper Error": np.maximum(0, upper_bound), 
    }
)

df_mean_shap = df_mean_shap.sort_values(by="Mean SHAP", ascending=True)

fig, ax = plt.subplots(figsize=(12, 10), dpi=160)

features = df_mean_shap["Feature"]
mean_shap_values = df_mean_shap["Mean SHAP"]
y_positions = np.arange(len(features)) * 2  # Space out the positions

ax.barh(
    y_positions,
    mean_shap_values,
    xerr=np.array([df_mean_shap["Lower Error"], df_mean_shap["Upper Error"]]),
    color="skyblue",
    capsize=3,
    edgecolor="black",
    alpha=0.8,
)

ax.set_yticks(y_positions)
ax.set_yticklabels(features, fontsize=12)

ax.set_xlabel("Mean SHAP Value (average impact on the outcome)", fontsize=14)
ax.set_ylabel("Features", fontsize=14)
ax.set_title("Global Feature Importance", fontsize=16)
ax.tick_params(axis="x", labelsize=12)
ax.grid(axis="x", linestyle="--", alpha=0.7)

plt.tight_layout()
output_path = r'D:\Figures\SHAP_simulation_grid.pdf'
plt.savefig(output_path, format="pdf", dpi=800, bbox_inches="tight")
plt.show()


In [ ]:
X_test = X_test.iloc[:5797].reset_index(drop=True)

l_95 = np.percentile(shap_bootstrap_values, axis=0, q=2.5)  
u_95 = np.percentile(shap_bootstrap_values, axis=0, q=97.5) 

assert l_95.shape == u_95.shape, "Shape mismatch: l_95 and u_95"
assert X_test.shape[0] == l_95.shape[0], "Shape mismatch: X_test rows and SHAP values"

def plot(ax, term=0):
   
    order = np.argsort(X_test.iloc[:, term])

    assert term < X_test.shape[1], f"Feature index {term} is out of bounds"

    ax.fill_between(
        X_test.iloc[:, term].values[order],
        l_95[order, term],
        u_95[order, term],
        color='lightblue',
        alpha=0.6
    )
    
    ax.scatter(
        X_test.iloc[:, term].values[order],
        shap_bootstrap_values.mean(axis=0)[order, term],
        s=10,
        color='black'
    )
    
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    
    ax.set_xlabel(X_test.columns[term], fontsize=13)
    ax.set_ylabel("SHAP value", fontsize=13)
    plt.tight_layout()

fig, ax = plt.subplots(3, 3, figsize=(12, 8), dpi=300) 
ax = ax.ravel()  

selected_features = [2, 7, 9, 8, 3, 5, 10, 4, 6]  

index = 0
for feature_idx in selected_features:
    if feature_idx >= X_test.shape[1]:
        print(f"Skipping feature index {feature_idx}: out of bounds")
        continue
    plot(ax=ax[index], term=feature_idx)
    index += 1

plt.tight_layout()
plt.savefig(r'D:\Figures\global_feature_importance_with_intervals_test_grid.pdf', format="pdf", dpi=800)
plt.show()